## Single Asset Bermudan using Neural Static Replication
This example builds a simple example of neural static replication for a Bermudan put option.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.optim import Adam
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.nn.init as init
from tqdm.notebook import trange

In [ ]:
torch.manual_seed(42)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Model Parameters
For simplicity to spot is normalized to 1.0

In [ ]:
S0 = 1.0
T = 1.0
N = 10
dt = T / N
r = 0.06
sigma = 0.2
M = 32768
hidden_dim = 64
epochs = 200
df = np.exp(-r * dt)
Tdf = np.exp(-r * T)
K = S0

### Monte Carlo
The model requires a set of simulated Monte Carlo paths

In [ ]:
def simulate_asset_paths(S0, r, sigma, T, dt, N, M, device=device):
    # Initialize the tensor to store paths on the specified device
    paths = torch.zeros((M, N + 1), device=device)
    
    # Set the initial asset price
    paths[:, 0] = S0
    
    # Precompute constants
    drift = (r - 0.5 * sigma ** 2) * dt
    diffusion = sigma * torch.sqrt(torch.tensor(dt, device=device))
    
    # Generate paths
    for t in range(1, N + 1):
        Z = torch.randn(M, device=device)
        paths[:, t] = paths[:, t - 1] * torch.exp(drift + diffusion * Z)
    
    return paths

In [ ]:
paths = simulate_asset_paths(S0, r, sigma, T, dt, N, M)

In [ ]:
strike = torch.full_like(paths[:,-1], K).to(device)

As a check, the European put price is calculated from the Monte Carlo paths

In [ ]:
values = torch.maximum(strike - paths[:,-1], torch.zeros_like(strike).to(device)).detach()

In [ ]:
torch.mean(values).cpu().item() * Tdf

### Black-Scholes Functions
The model needs a PyTorch implementation of the Black-Scholes function for continuation value

In [ ]:
def norm_cdf(x):
    return 0.5 * (1 + torch.erf(x / torch.sqrt(torch.tensor(2.0))))

def black_scholes(S, K, T, r, sigma, option_type='call', device=device):
    """
    Black-Scholes formula to calculate the option price using PyTorch on a specified device.
    
    Parameters:
    - S: Underlying asset price (Tensor)
    - K: Strike price (Tensor)
    - T: Time to maturity in years (Tensor)
    - r: Risk-free interest rate (Tensor)
    - sigma: Volatility (standard deviation of returns) (Tensor)
    - option_type: 'call' for call option, 'put' for put option
    - device: Device to perform the calculation on ('cpu' or 'cuda')

    Returns:
    - Option price (Tensor)
    """
    S = S.to(device)
    K = K.to(device)
    T = T.to(device)
    r = r.to(device)
    sigma = sigma.to(device)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    if option_type == 'call':
        return S * norm_cdf(d1) - K * torch.exp(-r * T) * norm_cdf(d2)
    elif option_type == 'put':
        return K * torch.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    else:
        raise ValueError("option_type must be either 'call' or 'put'")

In [ ]:
tS0 = torch.full((1,), S0)
tK = torch.full((1,), K)
tT = torch.full((1,), T)
tdt = torch.full((1,), dt)
tr = torch.full((1,), r)
tsigma = torch.full((1,), sigma)

The function is tested here for the same input values

In [ ]:
call_price = black_scholes(tS0, tK, tT, tr, tsigma)
put_price = black_scholes(tS0, tK, tT, tr, tsigma, 'put')
fwd = S0 * np.exp(r * T)
print('Call {}, Put {}, Fwd {}'.format(call_price.detach().cpu().item(), 
                                       put_price.detach().cpu().item(), fwd))

### Neural Static Replication
A simple network with one hidden layer is used.

In [ ]:
class PricingModel(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(PricingModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)
        init.normal_(self.fc1.weight, mean=0.0, std=0.01)
        init.normal_(self.fc2.weight, mean=0.0, std=0.01)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

The training loop is the standard code, with early stopping implemented.

In [ ]:
def train_model_with_early_stopping(model, train_loader, test_loader, loss_fn, 
                                    optimizer, epochs, patience=6):
    train_errors = []
    test_errors = []

    best_test_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X.unsqueeze(dim=-1))
            loss = loss_fn(outputs.squeeze(), batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)

        # Evaluation on the test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X.unsqueeze(dim=-1))
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.6f}, \
                                   Test Loss: {test_loss:.6f}")

        # Early stopping logic
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1} - Best test loss: {best_test_loss:.6f} at epoch {best_epoch+1}")
            break

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['best_epoch'] = best_epoch
    history['best_test_loss'] = best_test_loss
    return history

Data loaders are created on the fly inside the model loop and hence a function is needed to do this

In [ ]:
def create_dataloaders(input_tensor, target_tensor, val_percent=0.3, batch_size=256):
    """
    Create training and validation dataloaders after splitting and standard scaling.

    Parameters:
    - input_tensor: The input tensor (1D or 2D)
    - target_tensor: The target tensor (1D)
    - val_percent: The fraction of data to use for validation
    - batch_size: Batch size for the dataloaders

    Returns:
    - train_loader: DataLoader for training set
    - val_loader: DataLoader for validation set
    """
    # Step 1: Split into training and validation sets
    dataset = TensorDataset(input_tensor, target_tensor)
    val_size = int(len(dataset) * val_percent)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    # Extract training data for normalization
    train_inputs = torch.stack([x[0] for x in train_dataset])
    train_targets = torch.stack([x[1] for x in train_dataset])
    
    train_dataset = TensorDataset(train_inputs, train_targets)

    val_inputs = torch.stack([x[0] for x in val_dataset])
    val_targets = torch.stack([x[1] for x in val_dataset])
    val_dataset = TensorDataset(val_inputs, val_targets)

    # Step 3: Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

The continuation value is calculated using the model weights and biases, and weighted put and call values

In [ ]:
def bermudan_continuation_value(model, S_tm1, r, sigma, delta_t):
    """
    Calculate the continuation value of a Bermudan option.

    Parameters:
    - model: Trained PricingModel neural network
    - S_tm1: Asset price at time t_m-1 (Tensor)
    - r: Risk-free interest rate (float)
    - sigma: Volatility (float)
    - delta_t: Time increment between t_m and t_m1 (float)

    Returns:
    - Continuation value (Tensor)
    """
    # Extract weights and biases from the trained model
    w_i = model.fc1.weight.data
    b_i = model.fc1.bias.data
    omega_i = model.fc2.weight.data.squeeze()  # Assuming only one neuron in output layer
    b2 = model.fc2.bias.data.squeeze()  # Assuming only one bias in output layer

    df1 = torch.exp(r * delta_t)
    continuation_values = []

    for i in range(len(w_i)):
        if w_i[i] > 0 and b_i[i] > 0:  # Case 1
            forward_value = w_i[i] * S_tm1 * df1 + b_i[i]
            continuation_values.append(forward_value)
        elif w_i[i] > 0 and b_i[i] <= 0:  # Case 2
            strike_price = -b_i[i] / w_i[i]
            european_call_value = black_scholes(S_tm1, strike_price, delta_t, r, sigma, 'call', device=S_tm1.device)
            continuation_values.append(european_call_value * w_i[i])
        elif w_i[i] <= 0 and b_i[i] > 0:  # Case 3
            strike_price = -b_i[i] / w_i[i]
            european_put_value = black_scholes(S_tm1, strike_price, delta_t, r, sigma, 'put', device=S_tm1.device)
            continuation_values.append(european_put_value * -w_i[i])
        elif w_i[i] <= 0 and b_i[i] <= 0:  # Case 4
            continuation_values.append(torch.tensor(0.0, device=S_tm1.device))  # Continuation value is zero
    return sum(omega_i[i] * continuation_values[i] for i in range(len(continuation_values))) + b2

In [ ]:
model = PricingModel(1, hidden_dim).to(device)
optimizer = Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

The model walks backwards through time steps. Training the neural network against the values as targets and the using the trained weights and biases to estimate the continuation value.

In [ ]:
step_hist = dict()
values = torch.maximum(strike - paths[:,-1], torch.zeros_like(strike).to(device)).detach()
for t in range(N, 0, -1):
    print('Starting timestep: {}'.format(t))

    train_loader, val_loader = create_dataloaders(paths[:, t], values)
    hist = train_model_with_early_stopping(model, train_loader, val_loader, 
                                    loss_fn, optimizer, epochs)
    step_hist[t] = hist
    payoff = torch.maximum(strike - paths[:,t-1], torch.zeros_like(strike).to(device)).detach()
    cont_value = bermudan_continuation_value(model, paths[:, t-1], tr.to(device), 
                                             tsigma.to(device), tdt.to(device))
    values = torch.max(payoff, cont_value)

The final value is the discounted mean of the last step

In [ ]:
V0 = torch.mean(values).detach().cpu().item()

The final value

In [ ]:
V0

To compare a function is built to value the same option using the binomial tree in Quantlib

In [ ]:
import QuantLib as ql

def value_american_option(option_type, K, S, T, r, y, 
                          sigma, steps=200):
    maturity = int(T * 365)
    day_count = ql.Actual365Fixed()
    evaluation_date = ql.Settings.instance().evaluationDate
    
    # Create the option object
    if option_type.lower() == 'call':
        payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    elif option_type.lower() == 'put':
        payoff = ql.PlainVanillaPayoff(ql.Option.Put, K)
    else:
        raise ValueError("Invalid option type. Use 'call' or 'put'.")

    option = ql.VanillaOption(payoff, ql.AmericanExercise(evaluation_date, evaluation_date+maturity))
    
    # Set up the market data and the process
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    risk_free_curve = ql.YieldTermStructureHandle(ql.FlatForward(evaluation_date, ql.QuoteHandle(ql.SimpleQuote(r)), day_count))
    dividend_curve = ql.YieldTermStructureHandle(ql.FlatForward(evaluation_date, ql.QuoteHandle(ql.SimpleQuote(y)), day_count))
    volatility_curve = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(evaluation_date, ql.TARGET(), ql.QuoteHandle(ql.SimpleQuote(sigma)), day_count))
    
    process = ql.BlackScholesMertonProcess(spot_handle, dividend_curve, risk_free_curve, volatility_curve)
    
    # Use the binomial tree method
    binomial_engine = ql.BinomialVanillaEngine(process, 'CRR', steps)
    option.setPricingEngine(binomial_engine)
    
    # Calculate the option price
    return option.NPV()

In [ ]:
qlput = value_american_option('put', K, S0, T, r, 0.0, sigma, N)

In [ ]:
qlput